In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

import torch
import json
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

from notebooks.local.utils import get_paths, create_folders, load_progress, save_progress, mark_done, is_done

PATHS    = get_paths()
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
THRESHOLDS = [0.05, 0.10, 0.20]

TEMPERATURE  = 15.0
N_BLOCKS     = 2
LR           = 1e-4
NUM_EPOCHS   = 10
BATCH_SIZE   = 1

create_folders(PATHS)
print("Device:", DEVICE, "  fp16:", USE_FP16)


In [ ]:
import os
import shutil, tarfile
from notebooks.local.utils import get_paths, create_folders, download_file

PATHS    = get_paths()
create_folders(PATHS)

spair_check = os.path.join(PATHS['spair71k'], 'JPEGImages')
if not os.path.exists(spair_check):
    print('SPair-71k not found. Downloading (~2 GB) ...')
    tar_path = os.path.join(PATHS['data'], 'SPair-71k.tar.gz')
    download_file(
        'http://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz',
        tar_path, desc='SPair-71k',
    )
    print('Extracting ...')
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall(PATHS['spair71k'])
    extracted_sub = os.path.join(PATHS['spair71k'], 'SPair-71k')
    if os.path.isdir(extracted_sub):
        for item in os.listdir(extracted_sub):
            shutil.move(os.path.join(extracted_sub, item),
                        os.path.join(PATHS['spair71k'], item))
        os.rmdir(extracted_sub)
    os.remove(tar_path)
    print('SPair-71k ready.')
else:
    print('SPair-71k already present.')

if not os.path.exists(PATHS['dinov2_w']):
    print('Downloading DINOv2 ViT-B/14 weights (~330 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth',
        PATHS['dinov2_w'], desc='DINOv2',
    )
else:
    print('DINOv2 weights present.')

if not os.path.exists(PATHS['sam_w']):
    print('Downloading SAM ViT-B weights (~370 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        PATHS['sam_w'], desc='SAM',
    )
else:
    print('SAM weights present.')

if not os.path.exists(PATHS['dinov3_w']):
    print('WARNING: DINOv3 weights not found at', PATHS['dinov3_w'])
    print('  Place dinov3_vitb16_pretrain.pth in weights/ (obtain from project maintainer).')
else:
    print('DINOv3 weights present.')


In [ ]:
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as vit_base_v2
from src.models.dinov3.dinov3.models.vision_transformer import vit_base as vit_base_v3
from src.models.segment_anything.segment_anything import sam_model_registry
from src.features.extractor import (
    extract_dense_features, extract_dense_features_SAM,
    pixel_to_patch_coord, patch_to_pixel_coord,
)
from src.matching.strategies import find_best_match_argmax
from src.metrics.pck import compute_pck_spair71k
from experiments.finetune import (
    freeze_model, unfreeze_last_n_blocks,
    compute_cross_entropy_loss, validate,
)
import torch.nn.functional as F
from notebooks.local.utils import safe_cosine_similarity, prepare_image


def load_pretrained(backbone, paths, device, use_fp16=False):
    if backbone == 'dinov2':
        model = vit_base_v2(
            img_size=(518, 518), patch_size=14,
            num_register_tokens=0, block_chunks=0, init_values=1.0,
        )
        ckpt = torch.load(paths['dinov2_w'], map_location=device, weights_only=True)
        model.load_state_dict(ckpt, strict=True)
        img_size, patch_size = 518, 14
    elif backbone == 'dinov3':
        model = vit_base_v3(img_size=512, patch_size=16, n_storage_tokens=4, mask_k_bias=True, layerscale_init=1.0e-05, norm_layer="layernormbf16")
        ckpt = torch.load(paths['dinov3_w'], map_location=device, weights_only=True)
        model.load_state_dict(ckpt, strict=True)
        img_size, patch_size = 512, 16
    elif backbone == 'sam':
        model = sam_model_registry['vit_b'](checkpoint=paths['sam_w'])
        img_size, patch_size = 512, 16
    model = model.to(device)
    if use_fp16 and backbone != 'sam':
        model = model.half()
    model.eval()
    return model, img_size, patch_size


def unfreeze_last_n_blocks_sam(model, n_blocks):
    enc = model.image_encoder
    total = len(enc.blocks)
    for i in range(total - n_blocks, total):
        for p in enc.blocks[i].parameters():
            p.requires_grad = True
    for p in enc.neck.parameters():
        p.requires_grad = True


def train_epoch_fp16(model, dataloader, optimizer, scaler, device,
                     img_size, patch_size, temperature, scheduler=None,
                     is_sam=False):
    model.train()
    total_loss, n = 0.0, 0
    for idx, sample in enumerate(dataloader):
        src_t = sample['src_img'].to(device)
        tgt_t = sample['trg_img'].to(device)
        src_t = F.interpolate(src_t, size=(img_size, img_size), mode='bilinear', align_corners=False)
        tgt_t = F.interpolate(tgt_t, size=(img_size, img_size), mode='bilinear', align_corners=False)
        if USE_FP16 and not is_sam:
            src_t = src_t.half()
            tgt_t = tgt_t.half()

        src_orig = (sample['src_imsize'][2], sample['src_imsize'][1])
        tgt_orig = (sample['trg_imsize'][2], sample['trg_imsize'][1])
        src_kps = sample['src_kps'].numpy()[0]
        trg_kps = sample['trg_kps'].numpy()[0]

        with autocast(enabled=USE_FP16):
            if is_sam:
                src_feat = extract_dense_features_SAM(model, src_t, training=True, image_size=img_size)
                tgt_feat = extract_dense_features_SAM(model, tgt_t, training=True, image_size=img_size)
            else:
                src_feat = extract_dense_features(model, src_t, training=True)
                tgt_feat = extract_dense_features(model, tgt_t, training=True)

            loss = compute_cross_entropy_loss(
                src_feat.float(), tgt_feat.float(),
                src_kps, trg_kps, src_orig, tgt_orig,
                img_size, patch_size, temperature,
            )

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        n += 1
        if (idx + 1) % 100 == 0:
            print(f"  batch {idx+1}/{len(dataloader)}  loss={loss.item():.4f}")
    return total_loss / max(n, 1)


def train_backbone_fp16(model, backbone, train_ds, val_ds, n_blocks,
                         lr, temperature, num_epochs, img_size, patch_size,
                         best_ckpt_path, is_sam=False):
    freeze_model(model)
    if is_sam:
        unfreeze_last_n_blocks_sam(model, n_blocks)
    else:
        unfreeze_last_n_blocks(model, n_blocks)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr)

    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    total_steps = num_epochs * len(loader)
    warmup_steps = min(100, total_steps // 10)
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1/max(warmup_steps,1), end_factor=1.0, total_iters=warmup_steps)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(total_steps - warmup_steps, 1))
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_steps])

    scaler = GradScaler(enabled=USE_FP16)
    best_pck, patience_count, patience = 0.0, 0, 2
    history = []

    for epoch in range(1, num_epochs + 1):
        train_loss = train_epoch_fp16(model, loader, optimizer, scaler, DEVICE,
                                      img_size, patch_size, temperature, scheduler, is_sam)
        val_pck = validate(model, val_ds, DEVICE, img_size, patch_size)
        lr_now = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch}: loss={train_loss:.4f}  val_pck@0.10={val_pck:.2f}%  lr={lr_now:.2e}")
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_pck': val_pck, 'lr': lr_now})

        if val_pck > best_pck:
            best_pck = val_pck
            patience_count = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_pck': val_pck,
                        'epoch': epoch}, best_ckpt_path)
            print(f"  => Saved best: {val_pck:.2f}%")
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    return best_pck, history


In [ ]:
from src.datasets.spair_dataset import SPairDataset

pair_ann = os.path.join(PATHS['spair71k'], 'PairAnnotation')
layout   = os.path.join(PATHS['spair71k'], 'Layout')
images   = os.path.join(PATHS['spair71k'], 'JPEGImages')

train_dataset = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'trn')
val_dataset   = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'val')
test_dataset  = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'test')
print(f"train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")


## Ablation 1 — Temperature

In [ ]:
from experiments.evaluate import evaluate, save_results

TEMPERATURES = [5.0, 10.0, 15.0, 20.0, 30.0]
PROGRESS_PATH = os.path.join(PATHS['step2_abl'], 'progress_temp.json')
progress = load_progress(PROGRESS_PATH)

for temp in TEMPERATURES:
    for backbone in ['dinov2', 'dinov3']:
        key = f"{backbone}_temp{temp}"
        if is_done(progress, backbone, 'temp_ablation', temp):
            print(f"Skip {key}")
            continue

        out_dir = os.path.join(PATHS['step2_abl'], f"{backbone}_temp{temp}")
        os.makedirs(out_dir, exist_ok=True)

        model, img_size, patch_size = load_pretrained(backbone, PATHS, DEVICE, USE_FP16)
        freeze_model(model)
        unfreeze_last_n_blocks(model, N_BLOCKS)

        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad], lr=LR)
        loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
        scaler = GradScaler(enabled=USE_FP16)

        # 1-epoch ablation
        train_epoch_fp16(model, loader, optimizer, scaler, DEVICE,
                         img_size, patch_size, temp, is_sam=False)
        val_pck = validate(model, val_dataset, DEVICE, img_size, patch_size)
        result = {'temperature': temp, 'val_pck@0.10': val_pck}
        with open(os.path.join(out_dir, 'result.json'), 'w') as f:
            json.dump(result, f, indent=2)
        print(f"{key}: val_pck@0.10={val_pck:.2f}%")
        mark_done(progress, backbone, 'temp_ablation', temp, PROGRESS_PATH)
        del model; torch.cuda.empty_cache()

print("Temperature ablation done.")


## Ablation 2 — Number of Unfrozen Blocks

In [ ]:
N_BLOCKS_LIST = [1, 2, 3, 4]
PROGRESS_PATH = os.path.join(PATHS['step2_abl'], 'progress_blocks.json')
progress = load_progress(PROGRESS_PATH)

for n_bl in N_BLOCKS_LIST:
    for backbone in ['dinov2', 'dinov3']:
        if is_done(progress, backbone, 'blocks_ablation', n_bl):
            print(f"Skip {backbone} blocks={n_bl}")
            continue

        out_dir = os.path.join(PATHS['step2_abl'], f"{backbone}_blocks{n_bl}")
        os.makedirs(out_dir, exist_ok=True)

        model, img_size, patch_size = load_pretrained(backbone, PATHS, DEVICE, USE_FP16)
        freeze_model(model)
        unfreeze_last_n_blocks(model, n_bl)
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad], lr=LR)
        loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
        scaler = GradScaler(enabled=USE_FP16)

        train_epoch_fp16(model, loader, optimizer, scaler, DEVICE,
                         img_size, patch_size, TEMPERATURE, is_sam=False)
        val_pck = validate(model, val_dataset, DEVICE, img_size, patch_size)
        result = {'n_blocks': n_bl, 'val_pck@0.10': val_pck}
        with open(os.path.join(out_dir, 'result.json'), 'w') as f:
            json.dump(result, f, indent=2)
        print(f"{backbone} blocks={n_bl}: val_pck@0.10={val_pck:.2f}%")
        mark_done(progress, backbone, 'blocks_ablation', n_bl, PROGRESS_PATH)
        del model; torch.cuda.empty_cache()

print("Blocks ablation done.")


## Ablation 3 — Learning Rate

In [ ]:
LR_LIST = [5e-5, 1e-4, 2e-4, 5e-4]
PROGRESS_PATH = os.path.join(PATHS['step2_abl'], 'progress_lr.json')
progress = load_progress(PROGRESS_PATH)

for lr in LR_LIST:
    for backbone in ['dinov2', 'dinov3']:
        lr_key = str(lr)
        if is_done(progress, backbone, 'lr_ablation', lr_key):
            print(f"Skip {backbone} lr={lr}")
            continue

        out_dir = os.path.join(PATHS['step2_abl'], f"{backbone}_lr{lr}")
        os.makedirs(out_dir, exist_ok=True)

        model, img_size, patch_size = load_pretrained(backbone, PATHS, DEVICE, USE_FP16)
        freeze_model(model)
        unfreeze_last_n_blocks(model, N_BLOCKS)
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad], lr=lr)
        loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
        scaler = GradScaler(enabled=USE_FP16)

        train_epoch_fp16(model, loader, optimizer, scaler, DEVICE,
                         img_size, patch_size, TEMPERATURE, is_sam=False)
        val_pck = validate(model, val_dataset, DEVICE, img_size, patch_size)
        result = {'lr': lr, 'val_pck@0.10': val_pck}
        with open(os.path.join(out_dir, 'result.json'), 'w') as f:
            json.dump(result, f, indent=2)
        print(f"{backbone} lr={lr}: val_pck@0.10={val_pck:.2f}%")
        mark_done(progress, backbone, 'lr_ablation', lr_key, PROGRESS_PATH)
        del model; torch.cuda.empty_cache()

print("LR ablation done.")


## Full Fine-tuning — DINOv2

In [ ]:
ckpt_path = PATHS['dinov2_ft']
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

if os.path.exists(ckpt_path):
    print(f"DINOv2 fine-tuned checkpoint already exists: {ckpt_path}")
else:
    model, img_size, patch_size = load_pretrained('dinov2', PATHS, DEVICE, USE_FP16)
    best_pck, history = train_backbone_fp16(
        model, 'dinov2', train_dataset, val_dataset,
        n_blocks=N_BLOCKS, lr=LR, temperature=TEMPERATURE,
        num_epochs=NUM_EPOCHS, img_size=img_size, patch_size=patch_size,
        best_ckpt_path=ckpt_path, is_sam=False,
    )
    print(f"DINOv2 fine-tuning done. Best val PCK@0.10: {best_pck:.2f}%")
    del model; torch.cuda.empty_cache()


## Full Fine-tuning — DINOv3

In [ ]:
ckpt_path = PATHS['dinov3_ft']
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

if os.path.exists(ckpt_path):
    print(f"DINOv3 fine-tuned checkpoint already exists: {ckpt_path}")
else:
    model, img_size, patch_size = load_pretrained('dinov3', PATHS, DEVICE, USE_FP16)
    best_pck, history = train_backbone_fp16(
        model, 'dinov3', train_dataset, val_dataset,
        n_blocks=N_BLOCKS, lr=LR, temperature=TEMPERATURE,
        num_epochs=NUM_EPOCHS, img_size=img_size, patch_size=patch_size,
        best_ckpt_path=ckpt_path, is_sam=False,
    )
    print(f"DINOv3 fine-tuning done. Best val PCK@0.10: {best_pck:.2f}%")
    del model; torch.cuda.empty_cache()


## Full Fine-tuning — SAM

In [ ]:
ckpt_path = PATHS['sam_ft']
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

if os.path.exists(ckpt_path):
    print(f"SAM fine-tuned checkpoint already exists: {ckpt_path}")
else:
    model, img_size, patch_size = load_pretrained('sam', PATHS, DEVICE, False)
    best_pck, history = train_backbone_fp16(
        model, 'sam', train_dataset, val_dataset,
        n_blocks=N_BLOCKS, lr=LR, temperature=TEMPERATURE,
        num_epochs=NUM_EPOCHS, img_size=img_size, patch_size=patch_size,
        best_ckpt_path=ckpt_path, is_sam=True,
    )
    print(f"SAM fine-tuning done. Best val PCK@0.10: {best_pck:.2f}%")
    del model; torch.cuda.empty_cache()


## Evaluation on SPair-71k Test

In [ ]:
from experiments.evaluate import evaluate, save_results

results = {}
for backbone, ft_path, img_size, patch_size in [
    ('dinov2', PATHS['dinov2_ft'], 518, 14),
    ('dinov3', PATHS['dinov3_ft'], 512, 16),
]:
    out_dir = os.path.join(PATHS['step2'], f"{backbone}_finetuned")
    os.makedirs(out_dir, exist_ok=True)
    stats_path = os.path.join(out_dir, 'overall_stats.json')

    if os.path.exists(stats_path):
        print(f"{backbone} eval — already done.")
        with open(stats_path) as f:
            results[backbone] = json.load(f)
        continue

    if not os.path.exists(ft_path):
        print(f"No checkpoint for {backbone}, skipping eval.")
        continue

    if backbone == 'dinov2':
        model = vit_base_v2(
            img_size=(518,518), patch_size=14,
            num_register_tokens=0, block_chunks=0, init_values=1.0)
    else:
        model = vit_base_v3(img_size=512, patch_size=16, n_storage_tokens=4, mask_k_bias=True, layerscale_init=1.0e-05, norm_layer="layernormbf16")
    ckpt = torch.load(ft_path, map_location=DEVICE, weights_only=True)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state, strict=True)
    model = model.to(DEVICE)
    if USE_FP16:
        model = model.half()
    model.eval()

    per_img, all_kp, t = evaluate(model, test_dataset, DEVICE, THRESHOLDS)
    save_results(per_img, all_kp, out_dir, t, THRESHOLDS)
    with open(stats_path) as f:
        results[backbone] = json.load(f)
    del model; torch.cuda.empty_cache()
    print(f"{backbone} eval done.")

print("Fine-tuned evaluation complete.")


## Summary

In [ ]:
rows = []
for backbone in ['dinov2', 'dinov3']:
    for variant in ['baseline', 'finetuned']:
        if variant == 'baseline':
            stats_path = os.path.join(PATHS['step1'], f'{backbone}_argmax', 'overall_stats.json')
        else:
            stats_path = os.path.join(PATHS['step2'], f'{backbone}_finetuned', 'overall_stats.json')
        if os.path.exists(stats_path):
            with open(stats_path) as f:
                s = json.load(f)
            rows.append({
                'Model': f"{backbone} {variant}",
                'PCK@0.05': round(s.get('pck@0.05',{}).get('mean', float('nan')), 2),
                'PCK@0.10': round(s.get('pck@0.10',{}).get('mean', float('nan')), 2),
                'PCK@0.20': round(s.get('pck@0.20',{}).get('mean', float('nan')), 2),
            })

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
